# BALoRA third-round inference audit

Replay the completed seed-42 checkpoints on the same six BTCV test volumes, 460 slices, and 2,466 oracle-point prompts. The notebook fits no segmentation model.

1. Attach the labeled 30-volume BTCV cohort and **BALoRA_Full_Run_Results(1).zip** or its extracted contents. This is the completed archive, SHA-256 `ccc8581ea2b5dfac4909056585448a9121f6e1d2a122352f6039d5680c185340`.
2. Select a CUDA GPU. Enable Internet for the pinned official SAM2 checkout and any missing dependencies. The verified backbone may also be attached as an input. Set exact paths in the next cell if you have multiple copies.
3. Run all cells. The full run uses the complete test coverage. To check setup on two slices, set `SCOPE = "smoke"` first. The smoke output is separate.
4. Download **BALoRA_Round3_Inference_Results.zip**. If paused, attach a Round 3 progress ZIP in a later session with the same accelerator and runtime and set `RESUME_PATH`. Completed slices are reused. Round 2 records use their original notebook. Rerunning after a caught error in the same session also reuses saved slices.

The first two arms compare learned and fixed decoder SD on the matched cohort. The remaining arms measure decoder MC dropout, frozen multimask selection by predicted IoU, and reference-based best Dice over the returned candidates. Each phase logs actual SAM2 substitutions with call and prompt denominators. Predicted-mask region scores and perimeter support a matched-label regional risk analysis. The paired encoder timing uses alternating model order after warm-up.

The archived cumulative ledger carries 46.3674 hours and records a 240-hour expanded cap. The original 30-hour cap was exceeded by the completed programme. This notebook appends new inference time and has a two-hour cooperative session allowance by default. Its estimates and outputs describe replay only. A complete full test requires all four arms. Actual GPU runtime was unverified when this notebook was prepared.

In [ ]:
from pathlib import Path
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import zipfile

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RESULT_PATH = ""            # Optional exact completed ZIP or extracted result directory
BTCV_ROOT = ""              # Optional exact labeled Training directory
BACKBONE_PATH = ""          # Optional local verified SAM2.1 Hiera Base+ .pt
RESUME_PATH = ""            # Optional previous Round 3 Results ZIP
SCOPE = "full"              # "smoke" runs two slices separately
SESSION_WORK_HOURS = 2.0
TOTAL_CUMULATIVE_HOURS = 240.0
RECONCILIATIONS = {}        # Only for an interrupted, unclosed prior session with host-log evidence

In [ ]:
PAYLOAD_SHA256 = "a7e374762e483e86a9f465c02e47372e188d0795640d49a39ec21a3296f8c0b5"
payload = base64.b64decode("""
...
""")
assert hashlib.sha256(payload).hexdigest() == PAYLOAD_SHA256
source = WORK / ("round3_source_" + PAYLOAD_SHA256[:12])
source.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert z.testzip() is None
    assert all((source / n).resolve().is_relative_to(source.resolve()) for n in z.namelist())
    z.extractall(source)
print(f"Verified replay source {PAYLOAD_SHA256[:12]}")

In [ ]:
output = WORK / ("BALoRA_Round3_Inference" if SCOPE == "full" else "BALoRA_Round3_Smoke")
settings = {
    "scope": SCOPE, "input_root": str(INPUT_ROOT), "output_root": str(output),
    "work_root": str(WORK / "round3_work"), "result_path": RESULT_PATH, "btcv_root": BTCV_ROOT,
    "backbone_path": BACKBONE_PATH, "resume_path": RESUME_PATH, "payload_sha256": PAYLOAD_SHA256,
    "session_work_hours": SESSION_WORK_HOURS, "total_cumulative_hours": TOTAL_CUMULATIVE_HOURS,
    "reconciliations": RECONCILIATIONS,
}
settings_path = source / ("settings_" + SCOPE + ".json")
temp_path = settings_path.with_suffix(".tmp")
temp_path.write_text(json.dumps(settings, indent=2))
os.replace(temp_path, settings_path)
args = [sys.executable, "-u", str(source / "inference_audit.py"), str(settings_path)]
with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as job:
    for line in job.stdout:
        print(line, end="", flush=True)
    code = job.wait()
status = output / "status.json"
if status.is_file():
    state = json.loads(status.read_text())
    print(json.dumps({k: state.get(k) for k in ("status", "full_cohort_complete", "coverage")}, indent=2))
if code:
    raise RuntimeError(f"Replay exited with code {code}. Review the traceback above and last_error.txt if written.")

The result ZIP includes `status.json`, `per_prompt.csv`, encoder_timing.json, resumable slice records, the cumulative ledger, and actual regenerated arrays and an example figure if the two selected pancreas slices were replayed. Use the overlap estimates only when `full_cohort_complete` is true. The figure illustrates the highest and lowest archived MI-ranked failures of the same organ and volume chosen by the documented rule. The examples have no prespecified detection threshold.